# Build the submission folder (CPU)

Creates `tts_project/project/` on Drive in the required layout (English/, Telugu/, Results/, Report/, README.md, requirements.txt), plus `project_submission.zip`.

1. Fill in the first cell (final checkpoints and speakers, and the shared link for the full checkpoints).
2. Run the second cell. Needs `inference.py`, `repo_patches.diff`, `telugu_symbols.diff`, `config_Svarah/`, `config_Telugu/`, `preprocessed_*.tar`, `output/ckpt/*` and `Results/English`, `Results/Telugu` on Drive.

In [1]:
EN_STEP, EN_SPK = 15000, "S000"
TE_STEP, TE_SPK = 22000, "TEM"
# Telugu checkpoint + speaker used for the FINAL Telugu audio
DRIVE_LINK = "https://drive.google.com/drive/folders/1a5j23s1Ur31dnGiNgBH_yArFlT_vwF1a?usp=drive_link"

In [2]:
import os, shutil, glob, re, json, subprocess, tarfile
os.chdir("/content")
from google.colab import drive
drive.mount('/content/drive')
import yaml
D = "/content/drive/MyDrive/tts_project"
PROJ = f"{D}/project"
shutil.rmtree(PROJ, ignore_errors=True); os.makedirs(PROJ)

IGN = shutil.ignore_patterns(".git", "demo", "img", "index.html", "preprocessed_data", "output", "__pycache__",
                             "*.zip", "generator_*.pth.tar", "raw_data")

def build_lang(lang_dir, dataset, diffs, cfg_dir, step, spk, extra=()):
    work = f"/content/build_{lang_dir}"
    shutil.rmtree(work, ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/ming024/FastSpeech2.git", work], check=True)
    for d in diffs:
        p = f"{D}/{d}"
        if os.path.exists(p): subprocess.run(["git", "apply", p], cwd=work, check=True)
        else: print("WARNING: missing patch file", d)
    dst = f"{PROJ}/{lang_dir}"
    shutil.copytree(work, dst, ignore=IGN)
    for other in glob.glob(f"{dst}/config/*"):                       # keep only this language's config
        if os.path.isdir(other) and os.path.basename(other) != dataset: shutil.rmtree(other)
    os.makedirs(f"{dst}/config/{dataset}", exist_ok=True)
    cfgs = {}
    for n in ["preprocess", "model", "train"]:
        shutil.copy(f"{D}/{cfg_dir}/{n}.yaml", f"{dst}/config/{dataset}/{n}.yaml")
        cfgs[n] = yaml.safe_load(open(f"{D}/{cfg_dir}/{n}.yaml"))
    yaml.safe_dump(cfgs, open(f"{dst}/config.yaml", "w"), sort_keys=False, allow_unicode=True)
    shutil.copy(f"{D}/inference.py", f"{dst}/inference.py")
    for f in extra:
        if os.path.exists(f"{D}/{f}"): shutil.copy(f"{D}/{f}", f"{dst}/{f}")
    # small metadata needed to rebuild/run the model: speaker map, pitch/energy stats, the exact train/val lists
    tmp = f"/content/pp_{dataset}"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    with tarfile.open(f"{D}/preprocessed_{dataset}.tar") as t:
        names = [f"{dataset}/{n}" for n in ["speakers.json", "stats.json", "train.txt", "val.txt"]]
        t.extractall(tmp, members=[m for m in t.getmembers() if m.name in names])
    os.makedirs(f"{dst}/preprocessed_data/{dataset}", exist_ok=True)
    for n in ["speakers.json", "stats.json", "train.txt", "val.txt"]:
        shutil.copy(f"{tmp}/{dataset}/{n}", f"{dst}/preprocessed_data/{dataset}/{n}")
    # inference-only checkpoint (weights only, fp16 for big tensors): small enough for a normal GitHub commit
    os.makedirs(f"{dst}/checkpoints", exist_ok=True)
    src = f"{D}/output/ckpt/{dataset}/{step}.pth.tar"
    if os.path.exists(src):
        import torch
        ck = torch.load(src, map_location="cpu", weights_only=False)
        sd = {k: (v.half() if v.is_floating_point() and v.numel() > 1000 else v) for k, v in ck["model"].items()}
        out = f"{dst}/checkpoints/{dataset}_step{step}_model_fp16.pth.tar"
        torch.save({"model": sd}, out)
        print(lang_dir, "slim checkpoint:", f"{os.path.getsize(out)/1e6:.1f} MB")
    else:
        print("WARNING: checkpoint not found:", src)
    open(f"{dst}/checkpoints/README.md", "w").write(
        f"# {lang_dir} checkpoints\n\n"
        f"* `{dataset}_step{step}_model_fp16.pth.tar`: weights only (large tensors stored in fp16), enough for inference "
        f"(`python inference.py ... --restore_step {step}` after renaming/copying it to `<ckpt_path>/{step}.pth.tar`).\n"
        f"* Full training checkpoints (weights + optimizer, about 418 MB each, every 2,000-3,000 steps) are in the shared folder: {DRIVE_LINK}\n")

build_lang("English", "Svarah", ["repo_patches.diff"], "config_Svarah", EN_STEP, EN_SPK,
           extra=["svarah_split.csv", "dict.txt"])
build_lang("Telugu", "Telugu", ["repo_patches.diff", "telugu_symbols.diff"], "config_Telugu", TE_STEP, TE_SPK,
           extra=["telugu_split.csv", "dict_ids.txt", "word_ids.tsv", "test_sentences_telugu.json"])

# results
for lang in ["English", "Telugu"]:
    if os.path.isdir(f"{D}/Results/{lang}"): shutil.copytree(f"{D}/Results/{lang}", f"{PROJ}/Results/{lang}")
    else: print("WARNING: no final Results folder for", lang)
os.makedirs(f"{PROJ}/Report", exist_ok=True)
open(f"{PROJ}/Report/PUT_Technical_Report.pdf_HERE.txt", "w").write("Place Technical_Report.pdf in this folder.\n")

# requirements (versions as installed in this environment)
import importlib.metadata as md
pk = ["torch", "numpy", "scipy", "librosa", "pyworld", "g2p_en", "nltk", "num2words", "tgt", "PyYAML", "scikit-learn",
      "matplotlib", "soundfile", "pypinyin", "Unidecode", "inflect", "tensorboard", "tqdm", "datasets"]
lines = []
for p in pk:
    try: lines.append(f"{p}=={md.version(p)}")
    except Exception: lines.append(p)
open(f"{PROJ}/requirements.txt", "w").write("\n".join(lines) + "\n# alignment (separate conda environment): montreal-forced-aligner==3.4.3\n")

README = f"""# FastSpeech 2 text-to-speech: English (Svarah) and Telugu (IndicTTS)

Two separate FastSpeech 2 models, built on https://github.com/ming024/FastSpeech2 (HiFi-GAN vocoder, Montreal Forced Aligner durations).

## Layout
* `English/`, `Telugu/`: the patched repository for each language (`train.py`, `preprocess.py`, `inference.py`, `config.yaml`, `config/<dataset>/*.yaml`, `checkpoints/`, plus `preprocessed_data/<dataset>/` with the speaker map, pitch/energy statistics and the exact train/validation lists).
* `Results/<language>/`: loss curves, generated test audio, and duration/pitch/energy plots per test sentence.
* `Report/Technical_Report.pdf`: implementation, data, training and analysis.

## Final models used for the results
* English: Svarah, checkpoint step {EN_STEP}, speaker `{EN_SPK}`.
* Telugu: IndicTTS Telugu, checkpoint step {TE_STEP}, speaker `{TE_SPK}`.

## Reproduce (outline)
1. Select/clean data and export audio + `.lab` files (see `svarah_split.csv` / `telugu_split.csv` for the exact selection and split).
2. Build the pronunciation dictionary (`dict.txt` for English; `dict_ids.txt` + `word_ids.tsv` for Telugu, where words are replaced by ids for the aligner).
3. Align: `mfa train --clean -j 2 --output_directory <TextGrid dir> <corpus> <dictionary> <model.zip>` (MFA 3.4.3).
4. Features: `python preprocess.py config/<dataset>/preprocess.yaml` (our patch uses the fixed split in `split_val_utts.txt`).
5. Train: `python train.py -p config/<dataset>/preprocess.yaml -m config/<dataset>/model.yaml -t config/<dataset>/train.yaml` (resume with `--restore_step N`).
6. Inference and plots: `python inference.py --lang english --dataset Svarah --restore_step {EN_STEP} --speaker {EN_SPK} --out Results/English` and `python inference.py --lang telugu --dataset Telugu --restore_step {TE_STEP} --speaker {TE_SPK} --sentences test_sentences_telugu.json --out Results/Telugu`.

Pretrained HiFi-GAN weights (`generator_universal.pth.tar`) come from the original repository (`hifigan/generator_universal.pth.tar.zip`); they are not duplicated here.

## Notes
* Full training checkpoints and the Colab notebooks used: {DRIVE_LINK}
* Datasets are used under their own licences (Svarah: AI4Bharat; IndicTTS Telugu: IIT Madras, via SPRINGLab on Hugging Face).
"""
open(f"{PROJ}/README.md", "w", encoding="utf-8").write(README)

# report: sizes, and anything too big for a normal GitHub push
big = []
for root, _, fs in os.walk(PROJ):
    for f in fs:
        p = os.path.join(root, f); s = os.path.getsize(p)
        if s > 95e6: big.append((p, s / 1e6))
tot = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(PROJ) for f in fs) / 1e6
print(f"project size: {tot:.0f} MB | files over 95 MB (GitHub limit 100 MB):", big or "none")
shutil.make_archive(f"{D}/project_submission", "zip", PROJ)
print("zip:", f"{D}/project_submission.zip")
for root, dirs, fs in os.walk(PROJ):
    depth = root[len(PROJ):].count(os.sep)
    if depth <= 2: print("  " * depth + os.path.basename(root) + "/", len(fs), "files")

Mounted at /content/drive


/tmp/ipykernel_462/3836212509.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(tmp, members=[m for m in t.getmembers() if m.name in names])


English slim checkpoint: 71.0 MB


/tmp/ipykernel_462/3836212509.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(tmp, members=[m for m in t.getmembers() if m.name in names])


Telugu slim checkpoint: 70.9 MB
project size: 166 MB | files over 95 MB (GitHub limit 100 MB): none
zip: /content/drive/MyDrive/tts_project/project_submission.zip
project/ 2 files
  English/ 14 files
    utils/ 2 files
    model/ 5 files
    text/ 6 files
    hifigan/ 4 files
    config/ 1 files
    transformer/ 6 files
    preprocessor/ 4 files
    audio/ 4 files
    lexicon/ 2 files
    preprocessed_data/ 0 files
    checkpoints/ 2 files
  Telugu/ 15 files
    utils/ 2 files
    model/ 5 files
    text/ 6 files
    hifigan/ 4 files
    config/ 1 files
    transformer/ 6 files
    preprocessor/ 4 files
    audio/ 4 files
    lexicon/ 2 files
    preprocessed_data/ 0 files
    checkpoints/ 2 files
  Results/ 0 files
    English/ 3 files
    Telugu/ 3 files
  Report/ 1 files


In [3]:
from google.colab import drive
drive.mount('/content/drive')
from IPython.display import Audio, display
D = "/content/drive/MyDrive/tts_project"
def texts(lang):
    out = {}
    for blk in open(f"{D}/Results/{lang}/phoneme_sequences.txt", encoding="utf-8").read().strip().split("\n\n"):
        L = blk.split("\n"); out[L[0].strip()] = L[1].replace("TEXT: ", "")
    return out
for lang, ids in [("English", ["E1", "E2a", "E2b", "E3", "E4"]), ("Telugu", ["T1", "T2a", "T2b", "T3", "T4"])]:
    t = texts(lang)
    for i in ids:
        print(f"--- {lang} {i}: {t[i]}")
        display(Audio(f"{D}/Results/{lang}/test_generated_audios/{i}.wav"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- English E1: The antidisestablishmentarianism movement gained considerable momentum during the parliamentary proceedings.


--- English E2a: The orchestra's conduct during the ceremony was praised by the audience.


--- English E2b: The researchers will conduct an experiment to evaluate the proposed method.


--- English E3: The rhyth m of the orchestra changed unexpectedly.


--- English E4: The meeting will start at 10:45 AM on 12 November 2026, so andaroo should arrive on time.


--- Telugu T1: న్యూరోఫార్మకోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.


--- Telugu T2a: తాడుపై అతని పట్టు చాలా బలంగా ఉంది.


--- Telugu T2b: ఆమె పట్టుచీరను ధరించి వేడుకకు వచ్చింది.


--- Telugu T3: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.


--- Telugu T4: ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి.
